# deep-graph-systemic-risk

**Author:** Sana Ur Rehman  
**Profession:** Data Scientist  
**Created:** 2026  

---

## License

This project is licensed under the **MIT License**. 

You are free to use, modify, distribute, and build upon this work for both commercial and non-commercial purposes, provided you give appropriate **credit** to the original author. For the full legal text and conditions, please refer to the `LICENSE` file included in this project's repository.

---

## Citation

If you reference or build upon this project, please provide appropriate credit.

For formal citation information, please see the project's `README.md` and `CITATION.cff` files.

## Purpose

This notebook prepares the Bank for International Settlements (BIS) Consolidated Banking Statistics bulk dataset for temporal graph-machine-learning experiments.

The analysis uses cross-border banking relationships as a directed, weighted network:

\[
\text{Reporting country} \rightarrow \text{Counterparty country}
\]

Each positive reported international claim is treated as a directed edge, and the claim amount is the edge weight.

The source file is:

```text
data/raw/WS_CBS_PUB_csv_col.csv
```

The goal is to create clean, documented, and reproducible datasets covering the full available history:

```text
1983-Q4 through 2026-Q1
```

The source frequency changes over time:

- The historical period before 2000 is primarily semiannual.
- The period from 2000-Q1 through 2026-Q1 is quarterly.

The preprocessing will preserve the actual available BIS periods. It will not create artificial quarterly observations for the earlier semiannual data.

## Selected banking-network definition

To maintain comparability with the previous project, the initial network is defined as:

- Measure: Amounts outstanding / Stocks.
- Reporting basis: Immediate counterparty basis.
- Balance-sheet position: International claims.
- CBS bank type: Domestic banks.
- Instrument type: All instruments.
- Remaining maturity: Total (all maturities).
- Currency type: All currencies.
- Counterparty sector: All sectors.

The raw file will remain unchanged. All cleaned files created in this notebook will be saved separately in `data/processed/`.

## Source

Bank for International Settlements. (2026). *Consolidated banking statistics* [Data set]. BIS Data Portal. Retrieved September 5, 2026, from https://data.bis.org/bulkdownload

Bank for International Settlements. (2026). *Consolidated banking statistics: Overview* [Data set documentation]. BIS Data Portal. Retrieved September 5, 2026, from https://www.bis.org/statistics/consstats.htm

Imports, paths, and raw-file check

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

raw_path = RAW_DIR / "WS_CBS_PUB_csv_col.csv"

print("Current working directory:", Path.cwd())
print("Raw data path:", raw_path)
print("Raw file exists:", raw_path.exists())

if raw_path.exists():
    print(f"Raw file size: {raw_path.stat().st_size / (1024 ** 2):,.2f} MB")
else:
    print("\nFile not found. Check the filename and confirm the notebook is inside the 'notebooks' folder.")

Current working directory: c:\Users\Yahya\Desktop\AI projects\deep-graph-systemic-risk\notebooks
Raw data path: ..\data\raw\WS_CBS_PUB_csv_col.csv
Raw file exists: True
Raw file size: 143.08 MB


Load data and inspect schema

In [2]:
bulk = pd.read_csv(raw_path, low_memory=False)

print("Raw dataset shape:", bulk.shape)
print(f"Memory used after loading: {bulk.memory_usage(deep=True).sum() / (1024 ** 2):,.2f} MB")

print("\nFirst 30 columns:")
for i, column in enumerate(bulk.columns[:30]):
    print(f"{i:>3}: {column}")

print("\nLast 15 columns:")
start_index = len(bulk.columns) - 15
for i, column in enumerate(bulk.columns[-15:], start=start_index):
    print(f"{i:>3}: {column}")

print("\nData types:")
display(
    bulk.dtypes
    .rename("dtype")
    .to_frame()
)

print("\nFirst five rows:")
display(bulk.head())

Raw dataset shape: (228370, 167)
Memory used after loading: 588.00 MB

First 30 columns:
  0: FREQ
  1: Frequency
  2: L_MEASURE
  3: Measure
  4: L_REP_CTY
  5: Reporting country
  6: CBS_BANK_TYPE
  7: CBS bank type
  8: CBS_BASIS
  9: CBS reporting basis
 10: L_POSITION
 11: Balance sheet position
 12: L_INSTR
 13: Type of instruments
 14: REM_MATURITY
 15: Remaining maturity
 16: CURR_TYPE_BOOK
 17: Currency type of booking location
 18: L_CP_SECTOR
 19: Counterparty sector
 20: L_CP_COUNTRY
 21: Counterparty country
 22: TIME_FORMAT
 23: Time Format
 24: COLLECTION
 25: Collection Indicator
 26: ORG_VISIBILITY
 27: Organisation visibility
 28: Series
 29: 1983-Q4

Last 15 columns:
152: 2022-Q3
153: 2022-Q4
154: 2023-Q1
155: 2023-Q2
156: 2023-Q3
157: 2023-Q4
158: 2024-Q1
159: 2024-Q2
160: 2024-Q3
161: 2024-Q4
162: 2025-Q1
163: 2025-Q2
164: 2025-Q3
165: 2025-Q4
166: 2026-Q1

Data types:


,dtype
FREQ,object
Frequency,object
L_MEASURE,object
Measure,object
L_REP_CTY,object
...,...
2025-Q1,float64
2025-Q2,float64
2025-Q3,float64
2025-Q4,float64



First five rows:


,FREQ,Frequency,L_MEASURE,Measure,L_REP_CTY,Reporting country,CBS_BANK_TYPE,CBS bank type,CBS_BASIS,CBS reporting basis,L_POSITION,Balance sheet position,L_INSTR,Type of instruments,REM_MATURITY,Remaining maturity,CURR_TYPE_BOOK,Currency type of booking location,L_CP_SECTOR,Counterparty sector,L_CP_COUNTRY,Counterparty country,TIME_FORMAT,Time Format,COLLECTION,Collection Indicator,ORG_VISIBILITY,Organisation visibility,Series,1983-Q4,1984-Q2,1984-Q4,1985-Q2,1985-Q4,1986-Q2,1986-Q4,1987-Q2,1987-Q4,1988-Q2,1988-Q4,1989-Q2,1989-Q4,1990-Q2,1990-Q4,1991-Q2,1991-Q4,1992-Q2,1992-Q4,1993-Q2,1993-Q4,...,2013-Q4,2014-Q1,2014-Q2,2014-Q3,2014-Q4,2015-Q1,2015-Q2,2015-Q3,2015-Q4,2016-Q1,2016-Q2,2016-Q3,2016-Q4,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3,2018-Q4,2019-Q1,2019-Q2,2019-Q3,2019-Q4,2020-Q1,2020-Q2,2020-Q3,2020-Q4,2021-Q1,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1
0,Q,Quarterly,S,Amounts outstanding / Stocks,GB,United Kingdom,4R,"Domestic banks(4B), excl. domestic positions",F,Immediate counterparty basis,I,International claims,A,All instruments,A,Total (all maturities),TO1,All currencies,C,Non-financial corporations,5J,All countries,NaN,NaN,E,End of period,E,Public,Q:S:GB:4R:F:I:A:A:TO1:C:5J,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,477336.0,491859.0,493979.0,467274.0,469785.0,448888.0,409017.0,392461.0,382249.0,379322.0,378926.0,374933.0,393285.0,422556.0,436107.0,457321.0,434353.0,403380.0,422104.0,403325.0,408696.0,396572.0,383403.0,402445.0,359913.0,362495.0,367140.0,381166.0,372748.0,394248.0,393563.0,415943.0,349594.0,339561.0,324249.0,349765.0,331914.00,383973.0,365957.0,403277.0,420417.0,422233.0,438012.0,407539.0,412279.0,418161.0,426598.0,449395.0,431520.0
1,Q,Quarterly,S,Amounts outstanding / Stocks,HK,Hong Kong SAR,4B,Domestic banks,U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,R,Non-bank private sector,VG,British Virgin Islands,NaN,NaN,E,End of period,E,Public,Q:S:HK:4B:U:C:A:A:TO1:R:VG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,GL,Greenland,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:GL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,IR,Iran,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:IR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,FO,Faeroe Islands,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:FO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-133.0,N

## Raw Dataset Schema Check

The BIS bulk file contains 228,370 rows and 167 columns.

The first 29 columns describe each statistical series, including:

- Frequency.
- Measure.
- Reporting country.
- CBS bank type.
- CBS reporting basis.
- Balance-sheet position.
- Instrument type.
- Remaining maturity.
- Currency type.
- Counterparty sector.
- Counterparty country.
- Series identifier.

The remaining columns contain period-specific observations. The earliest available period is `1983-Q4`, and the latest available period is `2026-Q1`.

The file therefore provides a long historical window, but the frequency is not uniform across the entire sample. Earlier observations are primarily semiannual, while quarterly observations are available from 2000 onward.

No values have been changed at this stage. The raw data has only been loaded and inspected.

identify metadata and period columns

In [3]:
metadata_columns = [
    "FREQ",
    "Frequency",
    "L_MEASURE",
    "Measure",
    "L_REP_CTY",
    "Reporting country",
    "CBS_BANK_TYPE",
    "CBS bank type",
    "CBS_BASIS",
    "CBS reporting basis",
    "L_POSITION",
    "Balance sheet position",
    "L_INSTR",
    "Type of instruments",
    "REM_MATURITY",
    "Remaining maturity",
    "CURR_TYPE_BOOK",
    "Currency type of booking location",
    "L_CP_SECTOR",
    "Counterparty sector",
    "L_CP_COUNTRY",
    "Counterparty country",
    "TIME_FORMAT",
    "Time Format",
    "COLLECTION",
    "Collection Indicator",
    "ORG_VISIBILITY",
    "Organisation visibility",
    "Series",
]

missing_metadata_columns = [
    column for column in metadata_columns
    if column not in bulk.columns
]

period_columns = [
    column for column in bulk.columns
    if isinstance(column, str)
    and len(column) == 7
    and column[4] == "-"
    and column[-2:] in {"Q1", "Q2", "Q3", "Q4"}
]

unexpected_columns = [
    column for column in bulk.columns
    if column not in metadata_columns + period_columns
]

print("Metadata columns:", len(metadata_columns))
print("Missing metadata columns:", missing_metadata_columns)
print("Period columns:", len(period_columns))
print("Unexpected columns:", unexpected_columns)

print("\nFirst 10 period columns:")
print(period_columns[:10])

print("\nLast 10 period columns:")
print(period_columns[-10:])

Metadata columns: 29
Missing metadata columns: []
Period columns: 138
Unexpected columns: []

First 10 period columns:
['1983-Q4', '1984-Q2', '1984-Q4', '1985-Q2', '1985-Q4', '1986-Q2', '1986-Q4', '1987-Q2', '1987-Q4', '1988-Q2']

Last 10 period columns:
['2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4', '2026-Q1']


## Period Coverage and Frequency

The raw BIS bulk dataset contains 138 valid period columns from `1983-Q4` through `2026-Q1`.

The historical coverage is not uniformly quarterly:

- From `1983-Q4` through `1999-Q4`, the available periods are primarily semiannual, generally Q2 and Q4.
- From `2000-Q1` through `2025-Q4`, the dataset provides quarterly observations.
- The most recent available period is `2026-Q1`.

The preprocessing will preserve these actual source periods. No artificial observations will be interpolated for missing historical quarters. This is important because synthetic quarterly values could distort temporal graph structure and create misleading training data for graph autoencoders or temporal graph neural networks.

Verify the frequency structure

In [4]:
def period_to_year_quarter(period):
    year = int(period[:4])
    quarter = period[-2:]
    return year, quarter

period_info = pd.DataFrame(
    [period_to_year_quarter(period) for period in period_columns],
    columns=["year", "quarter"]
)

period_frequency = (
    period_info
    .groupby("year")["quarter"]
    .agg(lambda quarters: ", ".join(quarters))
    .reset_index(name="available_periods")
)

period_frequency["n_periods"] = (
    period_info
    .groupby("year")
    .size()
    .values
)

print("Years covered:", f"{period_info['year'].min()} to {period_info['year'].max()}")
print("Number of source periods:", len(period_columns))

print("\nEarly historical period:")
display(period_frequency.head(20))

print("\nTransition years around 1998–2001:")
display(
    period_frequency[
        period_frequency["year"].between(1998, 2001)
    ]
)

print("\nLatest available years:")
display(period_frequency.tail(10))

print("\nNumber of years by observation frequency:")
display(
    period_frequency["n_periods"]
    .value_counts()
    .sort_index()
    .rename_axis("periods_per_year")
    .reset_index(name="number_of_years")
)

Years covered: 1983 to 2026
Number of source periods: 138

Early historical period:


,year,available_periods,n_periods
0,1983,Q4,1
1,1984,"Q2, Q4",2
2,1985,"Q2, Q4",2
3,1986,"Q2, Q4",2
4,1987,"Q2, Q4",2
5,1988,"Q2, Q4",2
6,1989,"Q2, Q4",2
7,1990,"Q2, Q4",2
8,1991,"Q2, Q4",2
9,1992,"Q2, Q4",2



Transition years around 1998–2001:


,year,available_periods,n_periods
15,1998,"Q2, Q4",2
16,1999,"Q2, Q4",2
17,2000,"Q1, Q2, Q3, Q4",4
18,2001,"Q1, Q2, Q3, Q4",4



Latest available years:


,year,available_periods,n_periods
34,2017,"Q1, Q2, Q3, Q4",4
35,2018,"Q1, Q2, Q3, Q4",4
36,2019,"Q1, Q2, Q3, Q4",4
37,2020,"Q1, Q2, Q3, Q4",4
38,2021,"Q1, Q2, Q3, Q4",4
39,2022,"Q1, Q2, Q3, Q4",4
40,2023,"Q1, Q2, Q3, Q4",4
41,2024,"Q1, Q2, Q3, Q4",4
42,2025,"Q1, Q2, Q3, Q4",4
43,2026,Q1,1



Number of years by observation frequency:


,periods_per_year,number_of_years
0,1,2
1,2,16
2,4,26


## Frequency Decision for Graph Modeling

The source contains 138 actual observation periods:

- `1983-Q4` is the first available observation.
- `1984-Q2` through `1999-Q4` are primarily semiannual observations.
- `2000-Q1` through `2025-Q4` are quarterly observations.
- `2026-Q1` is the latest available observation.

Two analytical windows will be preserved:

1. **Full historical window:** `1983-Q4` through `2026-Q1`, retaining every period supplied by BIS.
2. **Regular quarterly window:** `2000-Q1` through `2026-Q1`, containing a complete quarterly calendar apart from the incomplete current year.

The full historical file is useful for long-run context and robustness checks. The regular quarterly file is preferred for temporal graph models because its time steps are evenly spaced.

No interpolation is performed between semiannual periods. Interpolation could create artificial edges or artificial changes in edge weights and would contaminate anomaly-detection and temporal-prediction tasks.

Inspect key dimension values

In [5]:
key_dimensions = [
    "Measure",
    "CBS reporting basis",
    "Balance sheet position",
    "CBS bank type",
    "Type of instruments",
    "Remaining maturity",
    "Currency type of booking location",
    "Counterparty sector",
]

for column in key_dimensions:
    print("\n" + "=" * 90)
    print(column)
    print("=" * 90)

    summary = (
        bulk[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="row_count")
    )

    display(summary)


Measure


,Measure,row_count
0,Amounts outstanding / Stocks,179016
1,Break in stocks,49354



CBS reporting basis


,CBS reporting basis,row_count
0,Immediate counterparty basis,108574
1,Guarantor basis,85989
2,Net risk transfers (Inward-Outward),18243
3,Outward risk transfers,15564



Balance sheet position


...

,Remaining maturity,row_count
0,Total (all maturities),199329
1,Up to and including 1 year,24367
2,Over 2 years,2386
3,Over 1 year and up to and including 2 years,2288



Currency type of booking location


,Currency type of booking location,row_count
0,All currencies,217775
1,Local currency,10595



Counterparty sector


,Counterparty sector,row_count
0,All sectors,172019
1,Non-bank private sector,15866
2,"Banks, total",12590
3,Official sector,10289
4,Non-bank financial institutions,8499
5,Households and NPISHs,3108
6,Non-financial corporations,3039
7,Non-financial private sector,2960


## BIS Code-to-Label Validation

The BIS bulk file contains both short dimension codes and readable dimension labels. The codes are used for filtering because they are compact and stable within the downloaded data; the labels are retained for interpretation and reporting.

Before filtering, the code-to-label mappings are checked to ensure that the selected series represents the intended banking-network concept. This prevents accidental mixing of claims, risk transfers, guarantor-basis exposures, local positions, or other balance-sheet categories.

Validate selected codes

In [6]:
selected_code_labels = {
    "L_MEASURE": ("S", "Measure"),
    "CBS_BASIS": ("F", "CBS reporting basis"),
    "L_POSITION": ("I", "Balance sheet position"),
    "CBS_BANK_TYPE": ("4B", "CBS bank type"),
    "L_INSTR": ("A", "Type of instruments"),
    "REM_MATURITY": ("A", "Remaining maturity"),
    "CURR_TYPE_BOOK": ("TO1", "Currency type of booking location"),
    "L_CP_SECTOR": ("A", "Counterparty sector"),
}

code_label_validation = []

for code_column, (selected_code, label_column) in selected_code_labels.items():
    matching_labels = (
        bulk.loc[bulk[code_column] == selected_code, label_column]
        .dropna()
        .unique()
        .tolist()
    )

    code_label_validation.append({
        "code_column": code_column,
        "selected_code": selected_code,
        "label_column": label_column,
        "matching_labels": matching_labels,
        "code_exists": len(matching_labels) > 0,
    })

code_label_validation = pd.DataFrame(code_label_validation)

display(code_label_validation)

,code_column,selected_code,label_column,matching_labels,code_exists
0,L_MEASURE,S,Measure,[Amounts outstanding / Stocks],True
1,CBS_BASIS,F,CBS reporting basis,[Immediate counterparty basis],True
2,L_POSITION,I,Balance sheet position,[International claims],True
3,CBS_BANK_TYPE,4B,CBS bank type,[Domestic banks],True
4,L_INSTR,A,Type of instruments,[All instruments],True
5,REM_MATURITY,A,Remaining maturity,[Total (all maturities)],True
6,CURR_TYPE_BOOK,TO1,Currency type of booking location,[All currencies],True
7,L_CP_SECTOR,A,Counterparty sector,[All sectors],True


## Selecting the Network Series

The raw BIS file contains multiple statistical series. To construct a consistent temporal banking network, this notebook selects only the following combination:

- `Amounts outstanding / Stocks`: exposure levels rather than changes or breaks.
- `Immediate counterparty basis`: exposure assigned to the direct counterparty country.
- `International claims`: cross-border claims rather than local claims or liabilities.
- `Domestic banks`: the selected BIS banking-system category.
- `All instruments`: no restriction to a particular instrument type.
- `Total (all maturities)`: the complete maturity profile.
- `All currencies`: total exposure across booking currencies.
- `All sectors`: total exposure across counterparty sectors.

These filters are applied using BIS codes rather than text labels. The readable labels remain in the filtered dataframe so that later outputs are interpretable.

Apply the validated filters

In [7]:
selected_filters = {
    "L_MEASURE": "S",
    "CBS_BASIS": "F",
    "L_POSITION": "I",
    "CBS_BANK_TYPE": "4B",
    "L_INSTR": "A",
    "REM_MATURITY": "A",
    "CURR_TYPE_BOOK": "TO1",
    "L_CP_SECTOR": "A",
}

filtered_bulk = bulk.copy()

for column, selected_code in selected_filters.items():
    filtered_bulk = filtered_bulk[
        filtered_bulk[column] == selected_code
    ].copy()

print("Filtered dataset shape:", filtered_bulk.shape)

print("\nSelected labels after filtering:")
for code_column, selected_code in selected_filters.items():
    label_column = {
        "L_MEASURE": "Measure",
        "CBS_BASIS": "CBS reporting basis",
        "L_POSITION": "Balance sheet position",
        "CBS_BANK_TYPE": "CBS bank type",
        "L_INSTR": "Type of instruments",
        "REM_MATURITY": "Remaining maturity",
        "CURR_TYPE_BOOK": "Currency type of booking location",
        "L_CP_SECTOR": "Counterparty sector",
    }[code_column]

    labels = filtered_bulk[label_column].dropna().unique().tolist()
    print(f"{label_column}: {labels}")

print("\nFiltered-data preview:")
display(
    filtered_bulk[
        [
            "Series",
            "Reporting country",
            "Counterparty country",
            "Measure",
            "CBS reporting basis",
            "Balance sheet position",
            "CBS bank type",
            "Type of instruments",
            "Remaining maturity",
            "Currency type of booking location",
            "Counterparty sector",
        ]
    ].head(10)
)

Filtered dataset shape: (6514, 167)

Selected labels after filtering:
Measure: ['Amounts outstanding / Stocks']
CBS reporting basis: ['Immediate counterparty basis']
Balance sheet position: ['International claims']
CBS bank type: ['Domestic banks']
Type of instruments: ['All instruments']
Remaining maturity: ['Total (all maturities)']
Currency type of booking location: ['All currencies']
Counterparty sector: ['All sectors']

Filtered-data preview:


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector
1098,Q:S:GR:4B:F:I:A:A:TO1:A:2Z,Greece,Unallocated West Indies UK,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1122,Q:S:GR:4B:F:I:A:A:TO1:A:3P,Greece,All countries excluding residents,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1152,Q:S:GR:4B:F:I:A:A:TO1:A:1C,Greece,International organisations,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1156,Q:S:GR:4B:F:I:A:A:TO1:A:1E,Greece,Residents/Local,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1215,Q:S:GR:4B:F:I:A:A:TO1:A:1W,Greece,Unallocated British Overseas Territories,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1217,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1218,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1219,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1220,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1222,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors


## Inspecting Network Node Categories

The filtered dataset still contains aggregate, regional, domestic, and residual categories alongside named economies. These categories must be reviewed before constructing graph nodes.

Examples of categories that should not normally become individual country nodes include:

- `All reporting countries`.
- `All countries`.
- `All countries excluding residents`.
- `Residents/Local`.
- `International organisations`.
- `Euro area`.
- `Unallocated ...` categories.
- Other regional aggregates.

Named economies and territories are retained initially unless they are clearly aggregate or residual categories. The exclusion rule is explicit and documented rather than based on deleting rows arbitrarily.

Inspect all node categories

In [8]:
reporting_categories = (
    filtered_bulk["Reporting country"]
    .value_counts(dropna=False)
    .rename_axis("reporting_country")
    .reset_index(name="series_count")
)

counterparty_categories = (
    filtered_bulk["Counterparty country"]
    .value_counts(dropna=False)
    .rename_axis("counterparty_country")
    .reset_index(name="series_count")
)

print("Reporting-country categories:")
display(reporting_categories)

print("\nNumber of counterparty categories:",
      counterparty_categories["counterparty_country"].nunique())

print("\nCounterparty categories:")
display(counterparty_categories)

Reporting-country categories:


,reporting_country,series_count
0,All reporting countries,252
1,Spain,232
2,Netherlands,231
3,France,229
4,India,229
5,United States,229
6,Canada,228
7,Italy,228
8,United Kingdom,228
9,Switzerland,225



Number of counterparty categories: 252

Counterparty categories:


,counterparty_country,series_count
0,All countries excluding residents,33
1,Netherlands,33
2,Singapore,33
3,Sweden,33
4,Portugal,33
...,...,...
247,Former Czechoslovakia,1
248,Former Soviet Union,1
249,Former Yugoslavia,1
250,Unallocated offshore centres,1


identify aggregate and residual categories

In [9]:
non_country_keywords = [
    "all countries",
    "excluding residents",
    "residents/local",
    "international organisation",
    "international organization",
    "unallocated",
    "euro area",
    "advanced economies",
    "emerging economies",
    "emerging market",
    "regional",
    "offshore centres",
    "offshore centers",
]

all_counterparty_categories = (
    filtered_bulk["Counterparty country"]
    .dropna()
    .astype(str)
    .unique()
)

possible_non_country_categories = sorted([
    category
    for category in all_counterparty_categories
    if any(
        keyword in category.lower()
        for keyword in non_country_keywords
    )
])

print("Possible aggregate, regional, or residual counterparty categories:")
display(
    pd.DataFrame({
        "category": possible_non_country_categories
    })
)

print("\nNumber of possible non-country categories:",
      len(possible_non_country_categories))

Possible aggregate, regional, or residual counterparty categories:


,category
0,Advanced economies
1,All countries
2,All countries excluding residents
3,Emerging market and developing economies
4,Euro area
5,International organisations
6,Residents/Local
7,Unallocated British Overseas Territories
8,Unallocated West Indies UK
9,Unallocated advanced economies



Number of possible non-country categories: 16


## Excluding Aggregate and Residual Nodes

The country-level graph should contain named reporting and counterparty economies, not totals or geographic aggregates. The following counterparty categories are excluded:

- All-country totals.
- Regional groupings.
- Domestic/resident aggregates.
- International organizations.
- Unallocated categories.
- Offshore-centre aggregates.

Named economies and historical economy labels are retained unless they are explicitly classified as an aggregate or residual category. This conservative rule avoids deleting small economies or territories merely because they are not major financial centres.

The excluded categories remain present in the raw and filtered datasets and are removed only from the network-candidate dataset.

remove aggregate categories

In [10]:
excluded_counterparties = [
    "Advanced economies",
    "All countries",
    "All countries excluding residents",
    "Emerging market and developing economies",
    "Euro area",
    "International organisations",
    "Residents/Local",
    "Unallocated British Overseas Territories",
    "Unallocated West Indies UK",
    "Unallocated advanced economies",
    "Unallocated emerging Africa and Middle East",
    "Unallocated emerging Asia and Pacific",
    "Unallocated emerging Europe",
    "Unallocated emerging Latin America and Caribbean",
    "Unallocated location",
    "Unallocated offshore centres",
]

excluded_reporting_countries = [
    "All reporting countries"
]

network_candidates = filtered_bulk[
    ~filtered_bulk["Reporting country"].isin(
        excluded_reporting_countries
    )
    & ~filtered_bulk["Counterparty country"].isin(
        excluded_counterparties
    )
].copy()

print("Original filtered shape:", filtered_bulk.shape)
print("Network candidate shape:", network_candidates.shape)
print(
    "Reporting countries remaining:",
    network_candidates["Reporting country"].nunique()
)
print(
    "Counterparty categories remaining:",
    network_candidates["Counterparty country"].nunique()
)

remaining_suspicious = sorted(
    network_candidates.loc[
        network_candidates["Counterparty country"]
        .astype(str)
        .str.contains(
            "all countries|euro area|unallocated|"
            "residents/local|international organisations|"
            "advanced economies|emerging market",
            case=False,
            na=False
        ),
        "Counterparty country"
    ].unique()
)

print("\nRemaining suspicious counterparty categories:")
display(pd.DataFrame({
    "remaining_category": remaining_suspicious
}))

Original filtered shape: (6514, 167)
Network candidate shape: (6016, 167)
Reporting countries remaining: 32
Counterparty categories remaining: 226

Remaining suspicious counterparty categories:


,remaining_category


## Result of Node-Cleaning Step

The filtered BIS data originally contained 6,514 series. After removing the aggregate reporting category and 16 aggregate, regional, domestic, international-organization, offshore, and unallocated counterparty categories, 6,016 country-level series remain.

The cleaned candidate dataset contains:

- 32 reporting-country categories.
- 226 counterparty-country categories.
- No remaining categories matching the predefined aggregate or residual patterns.

The raw and initially filtered datasets are retained for audit purposes. Only the cleaned candidate dataset is used for subsequent period selection, reshaping, missingness analysis, and graph construction.

In [11]:
full_periods = period_columns.copy()

quarterly_periods = [
    period for period in full_periods
    if int(period[:4]) >= 2000
]

print("Full historical periods:")
print(f"Count: {len(full_periods)}")
print(f"First: {full_periods[0]}")
print(f"Last: {full_periods[-1]}")

print("\nRegular quarterly periods:")
print(f"Count: {len(quarterly_periods)}")
print(f"First: {quarterly_periods[0]}")
print(f"Last: {quarterly_periods[-1]}")

print("\nPeriods missing from the regular quarterly calendar:")
expected_quarterly_periods = [
    f"{year}-Q{quarter}"
    for year in range(2000, 2027)
    for quarter in range(1, 5)
]

missing_quarterly_periods = [
    period for period in expected_quarterly_periods
    if period not in quarterly_periods
]

print(missing_quarterly_periods)

Full historical periods:
Count: 138
First: 1983-Q4
Last: 2026-Q1

Regular quarterly periods:
Count: 105
First: 2000-Q1
Last: 2026-Q1

Periods missing from the regular quarterly calendar:
['2026-Q2', '2026-Q3', '2026-Q4']


## Creating Wide-Format Historical Datasets

Two clean wide-format datasets are created from the country-level network candidates:

1. **Full historical wide dataset:** retains all 138 BIS periods from `1983-Q4` through `2026-Q1`.
2. **Regular quarterly wide dataset:** retains the 105 regularly quarterly periods from `2000-Q1` through `2026-Q1`.

The wide files preserve one row per reporting-country/counterparty-country series and one column per source period. They are retained as transparent source-derived versions of the cleaned data.

The next preprocessing stage will reshape both versions into long format, where each row represents one country-to-country observation in one period.

Create wide datasets

In [12]:
network_metadata_columns = [
    "Series",
    "Reporting country",
    "Counterparty country",
    "Measure",
    "CBS reporting basis",
    "Balance sheet position",
    "CBS bank type",
    "Type of instruments",
    "Remaining maturity",
    "Currency type of booking location",
    "Counterparty sector",
]

network_full_wide = network_candidates[
    network_metadata_columns + full_periods
].copy()

network_quarterly_wide = network_candidates[
    network_metadata_columns + quarterly_periods
].copy()

print("Full historical wide shape:", network_full_wide.shape)
print("Regular quarterly wide shape:", network_quarterly_wide.shape)

print("\nFull historical period range:")
print(full_periods[0], "to", full_periods[-1])

print("\nRegular quarterly period range:")
print(quarterly_periods[0], "to", quarterly_periods[-1])

print("\nPreview of the full historical wide dataset:")
display(network_full_wide.head())

Full historical wide shape: (6016, 149)
Regular quarterly wide shape: (6016, 116)

Full historical period range:
1983-Q4 to 2026-Q1

Regular quarterly period range:
2000-Q1 to 2026-Q1

Preview of the full historical wide dataset:


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,1983-Q4,1984-Q2,1984-Q4,1985-Q2,1985-Q4,1986-Q2,1986-Q4,1987-Q2,1987-Q4,1988-Q2,1988-Q4,1989-Q2,1989-Q4,1990-Q2,1990-Q4,1991-Q2,1991-Q4,1992-Q2,1992-Q4,1993-Q2,1993-Q4,1994-Q2,1994-Q4,1995-Q2,1995-Q4,1996-Q2,1996-Q4,1997-Q2,1997-Q4,1998-Q2,1998-Q4,1999-Q2,1999-Q4,2000-Q1,2000-Q2,2000-Q3,2000-Q4,2001-Q1,2001-Q2,...,2013-Q4,2014-Q1,2014-Q2,2014-Q3,2014-Q4,2015-Q1,2015-Q2,2015-Q3,2015-Q4,2016-Q1,2016-Q2,2016-Q3,2016-Q4,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3,2018-Q4,2019-Q1,2019-Q2,2019-Q3,2019-Q4,2020-Q1,2020-Q2,2020-Q3,2020-Q4,2021-Q1,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1
1217,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1218,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1219,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1220,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1222,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,4.0,4.0,3.0,3.0,3.0,4.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,3.0,2.0,2.0,2.0,NaN,1.0,1.0,NaN,NaN,NaN,1.0,1.021,1.077,1.057,0.192,0.558,0.543,0.513,1.554,1.59,1.64,1.629


## Reshaping to Long Format

The wide datasets are reshaped into long format for temporal analysis.

In the long format:

- Each row represents one reporting-country/counterparty-country series in one period.
- `period` identifies the BIS observation period.
- `claim_value_usd_millions` stores the reported claim value.
- Metadata columns identify the series definition and the two network endpoints.

Two long-format datasets are created:

1. `network_full_long`: all 138 available historical periods.
2. `network_quarterly_long`: the 105-period regular quarterly window.

Missing values are retained at this stage for data-quality assessment. They are not interpreted as zero claims.

Reshape both datasets

In [13]:
network_full_long = network_full_wide.melt(
    id_vars=network_metadata_columns,
    value_vars=full_periods,
    var_name="period",
    value_name="claim_value_usd_millions"
)

network_quarterly_long = network_quarterly_wide.melt(
    id_vars=network_metadata_columns,
    value_vars=quarterly_periods,
    var_name="period",
    value_name="claim_value_usd_millions"
)

print("Full historical long shape:", network_full_long.shape)
print("Regular quarterly long shape:", network_quarterly_long.shape)

print("\nFull historical preview:")
display(network_full_long.head())

print("\nRegular quarterly preview:")
display(network_quarterly_long.head())

Full historical long shape: (830208, 13)
Regular quarterly long shape: (631680, 13)

Full historical preview:


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,period,claim_value_usd_millions
0,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,1983-Q4,NaN
1,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,1983-Q4,NaN
2,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,1983-Q4,NaN
3,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,1983-Q4,NaN
4,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,1983-Q4,NaN



Regular quarterly preview:


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,period,claim_value_usd_millions
0,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2000-Q1,NaN
1,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2000-Q1,NaN
2,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2000-Q1,NaN
3,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2000-Q1,NaN
4,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2000-Q1,NaN


## Time-Feature Engineering

The BIS `period` labels are converted into standardized time fields while preserving the original label.

The derived fields are:

- `year`: calendar year extracted from the BIS period.
- `quarter`: quarter number, from 1 to 4.
- `period_index`: sequential integer index based on the actual source period order.
- `is_quarterly_era`: `True` for periods from 2000-Q1 onward.
- `is_semiannual_era`: `True` for the historical semiannual period before 2000.

The `period_index` follows the order of observations supplied by BIS. It does not assume that every historical quarter exists, so it preserves the irregular semiannual-to-quarterly transition.

Add time fields

In [14]:
period_order = {
    period: index
    for index, period in enumerate(full_periods)
}

def add_time_features(dataframe):
    result = dataframe.copy()

    result["year"] = result["period"].str[:4].astype(int)
    result["quarter"] = result["period"].str[-1].astype(int)
    result["period_index"] = result["period"].map(period_order).astype(int)

    result["is_quarterly_era"] = result["year"] >= 2000
    result["is_semiannual_era"] = result["year"] < 2000

    return result

network_full_long = add_time_features(network_full_long)
network_quarterly_long = add_time_features(network_quarterly_long)

print("Full historical time-feature preview:")
display(
    network_full_long[
        ["period", "year", "quarter", "period_index",
         "is_quarterly_era", "is_semiannual_era"]
    ].drop_duplicates().head(10)
)

print("\nTransition-period preview:")
display(
    network_full_long[
        ["period", "year", "quarter", "period_index",
         "is_quarterly_era", "is_semiannual_era"]
    ]
    .drop_duplicates()
    .query("1998 <= year <= 2001")
)

print("\nRegular quarterly time-feature preview:")
display(
    network_quarterly_long[
        ["period", "year", "quarter", "period_index",
         "is_quarterly_era", "is_semiannual_era"]
    ].drop_duplicates().head(10)
)

Full historical time-feature preview:


,period,year,quarter,period_index,is_quarterly_era,is_semiannual_era
0,1983-Q4,1983,4,0,False,True
6016,1984-Q2,1984,2,1,False,True
12032,1984-Q4,1984,4,2,False,True
18048,1985-Q2,1985,2,3,False,True
24064,1985-Q4,1985,4,4,False,True
30080,1986-Q2,1986,2,5,False,True
36096,1986-Q4,1986,4,6,False,True
42112,1987-Q2,1987,2,7,False,True
48128,1987-Q4,1987,4,8,False,True
54144,1988-Q2,1988,2,9,False,True



Transition-period preview:


,period,year,quarter,period_index,is_quarterly_era,is_semiannual_era
174464,1998-Q2,1998,2,29,False,True
180480,1998-Q4,1998,4,30,False,True
186496,1999-Q2,1999,2,31,False,True
192512,1999-Q4,1999,4,32,False,True
198528,2000-Q1,2000,1,33,True,False
204544,2000-Q2,2000,2,34,True,False
210560,2000-Q3,2000,3,35,True,False
216576,2000-Q4,2000,4,36,True,False
222592,2001-Q1,2001,1,37,True,False
228608,2001-Q2,2001,2,38,True,False



Regular quarterly time-feature preview:


,period,year,quarter,period_index,is_quarterly_era,is_semiannual_era
0,2000-Q1,2000,1,33,True,False
6016,2000-Q2,2000,2,34,True,False
12032,2000-Q3,2000,3,35,True,False
18048,2000-Q4,2000,4,36,True,False
24064,2001-Q1,2001,1,37,True,False
30080,2001-Q2,2001,2,38,True,False
36096,2001-Q3,2001,3,39,True,False
42112,2001-Q4,2001,4,40,True,False
48128,2002-Q1,2002,1,41,True,False
54144,2002-Q2,2002,2,42,True,False


## Data Availability by Period

The BIS dataset is sparse: not every reporting-country/counterparty-country pair has a reported claim in every period.

The next checks measure, for every source period:

- Total potential country-pair series.
- Observed numerical claim values.
- Positive claim values.
- Negative claim values.
- Missing values.
- Percentage of country-pair series without a reported value.

Missing observations are not treated as zero claims. They remain missing in the audit datasets. Only positive reported claims will become edges in the graph dataset.

Period-level data-quality report

In [15]:
def make_period_quality_report(dataframe, period_order_map):
    quality = (
        dataframe
        .groupby(["period", "year", "quarter", "period_index"], as_index=False)
        .agg(
            total_series=("claim_value_usd_millions", "size"),
            observed_values=("claim_value_usd_millions", "count"),
            positive_values=(
                "claim_value_usd_millions",
                lambda values: (values > 0).sum()
            ),
            negative_values=(
                "claim_value_usd_millions",
                lambda values: (values < 0).sum()
            ),
            zero_values=(
                "claim_value_usd_millions",
                lambda values: (values == 0).sum()
            ),
        )
    )

    quality["missing_values"] = (
        quality["total_series"] - quality["observed_values"]
    )

    quality["missing_percent"] = (
        quality["missing_values"] / quality["total_series"] * 100
    ).round(2)

    quality["observed_percent"] = (
        quality["observed_values"] / quality["total_series"] * 100
    ).round(2)

    return quality.sort_values("period_index").reset_index(drop=True)

full_period_quality = make_period_quality_report(
    network_full_long,
    period_order
)

quarterly_period_quality = make_period_quality_report(
    network_quarterly_long,
    period_order
)

print("Full historical period-quality report:")
display(full_period_quality.head(12))

print("\nTransition period-quality report:")
display(
    full_period_quality[
        full_period_quality["year"].between(1998, 2001)
    ]
)

print("\nMost recent period-quality report:")
display(full_period_quality.tail(12))

print("\nFull historical missingness range:")
print(
    f"{full_period_quality['missing_percent'].min():.2f}% "
    f"to {full_period_quality['missing_percent'].max():.2f}%"
)

Full historical period-quality report:


,period,year,quarter,period_index,total_series,observed_values,positive_values,negative_values,zero_values,missing_values,missing_percent,observed_percent
0,1983-Q4,1983,4,0,6016,658,658,0,0,5358,89.06,10.94
1,1984-Q2,1984,2,1,6016,649,649,0,0,5367,89.21,10.79
2,1984-Q4,1984,4,2,6016,663,663,0,0,5353,88.98,11.02
3,1985-Q2,1985,2,3,6016,736,736,0,0,5280,87.77,12.23
4,1985-Q4,1985,4,4,6016,879,879,0,0,5137,85.39,14.61
5,1986-Q2,1986,2,5,6016,880,880,0,0,5136,85.37,14.63
6,1986-Q4,1986,4,6,6016,893,893,0,0,5123,85.16,14.84
7,1987-Q2,1987,2,7,6016,895,895,0,0,5121,85.12,14.88
8,1987-Q4,1987,4,8,6016,903,903,0,0,5113,84.99,15.01
9,1988-Q2,1988,2,9,6016,912,912,0,0,5104,84.84,15.16



Transition period-quality report:


,period,year,quarter,period_index,total_series,observed_values,positive_values,negative_values,zero_values,missing_values,missing_percent,observed_percent
29,1998-Q2,1998,2,29,6016,957,957,0,0,5059,84.09,15.91
30,1998-Q4,1998,4,30,6016,998,998,0,0,5018,83.41,16.59
31,1999-Q2,1999,2,31,6016,1131,1130,1,0,4885,81.20,18.80
32,1999-Q4,1999,4,32,6016,1275,1274,1,0,4741,78.81,21.19
33,2000-Q1,2000,1,33,6016,1271,1270,1,0,4745,78.87,21.13
34,2000-Q2,2000,2,34,6016,1273,1272,1,0,4743,78.84,21.16
35,2000-Q3,2000,3,35,6016,1180,1180,0,0,4836,80.39,19.61
36,2000-Q4,2000,4,36,6016,1418,1418,0,0,4598,76.43,23.57
37,2001-Q1,2001,1,37,6016,1400,1400,0,0,4616,76.73,23.27
38,2001-Q2,2001,2,38,6016,1415,1414,1,0,4601,76.48,23.52



Most recent period-quality report:


,period,year,quarter,period_index,total_series,observed_values,positive_values,negative_values,zero_values,missing_values,missing_percent,observed_percent
126,2023-Q2,2023,2,126,6016,2540,2536,4,0,3476,57.78,42.22
127,2023-Q3,2023,3,127,6016,2507,2503,4,0,3509,58.33,41.67
128,2023-Q4,2023,4,128,6016,2512,2504,8,0,3504,58.24,41.76
129,2024-Q1,2024,1,129,6016,2551,2545,6,0,3465,57.60,42.40
130,2024-Q2,2024,2,130,6016,2547,2542,5,0,3469,57.66,42.34
131,2024-Q3,2024,3,131,6016,2526,2521,5,0,3490,58.01,41.99
132,2024-Q4,2024,4,132,6016,2525,2521,4,0,3491,58.03,41.97
133,2025-Q1,2025,1,133,6016,2518,2515,3,0,3498,58.14,41.86
134,2025-Q2,2025,2,134,6016,2515,2509,6,0,3501,58.19,41.81
135,2025-Q3,2025,3,135,6016,2536,2529,7,0,3480,57.85,42.15



Full historical missingness range:
56.13% to 89.21%


## Coverage Result and Modeling Implications

The number of observed positive country-pair claims rises over the historical sample. Early snapshots contain roughly 650–900 positive reported edges, while recent quarterly snapshots contain approximately 2,500 positive reported edges.

The missing-value rate ranges from 56.13% to 89.21% across the full historical period. This pattern reflects changing reporting coverage, country availability, and BIS data detail over time. It must not be interpreted directly as a change in the real-world absence of banking relationships.

For graph analysis:

- Positive reported claims are used as directed weighted edges.
- Missing values are retained as unobserved in audit datasets and are not converted to zero claims.
- Period-level coverage statistics are retained and should be used when interpreting graph density, anomaly scores, or structural changes.
- The regular quarterly window from `2000-Q1` to `2026-Q1` is the main input for temporal graph models.
- The full historical period from `1983-Q4` to `2026-Q1` is retained for long-run descriptive analysis, robustness checks, and potential pretraining.

The analysis will avoid interpreting increases in the number of observed edges as purely economic changes without considering changes in reporting coverage.

clean values and inspect negatives

In [16]:
def value_quality_summary(dataframe, dataset_name):
    values = dataframe["claim_value_usd_millions"]

    summary = {
        "dataset": dataset_name,
        "total_rows": len(dataframe),
        "observed_values": values.notna().sum(),
        "missing_values": values.isna().sum(),
        "positive_values": (values > 0).sum(),
        "negative_values": (values < 0).sum(),
        "zero_values": (values == 0).sum(),
        "minimum_observed_value": values.min(),
        "median_observed_value": values.median(),
        "maximum_observed_value": values.max(),
    }

    return pd.DataFrame([summary])

value_quality = pd.concat([
    value_quality_summary(network_full_long, "full_historical"),
    value_quality_summary(network_quarterly_long, "regular_quarterly"),
], ignore_index=True)

display(value_quality)

negative_full = network_full_long[
    network_full_long["claim_value_usd_millions"] < 0
].copy()

print("Number of negative observations in the full historical dataset:",
      len(negative_full))

display(
    negative_full[
        [
            "Reporting country",
            "Counterparty country",
            "period",
            "claim_value_usd_millions",
        ]
    ]
    .sort_values("claim_value_usd_millions")
    .head(20)
)

,dataset,total_rows,observed_values,missing_values,positive_values,negative_values,zero_values,minimum_observed_value,median_observed_value,maximum_observed_value
0,full_historical,830208,249073,581135,248839,234,0,-1069.0,111.000,2098198.5
1,regular_quarterly,631680,219231,412449,218999,232,0,-1069.0,121.646,2098198.5


Number of negative observations in the full historical dataset: 234


,Reporting country,Counterparty country,period,claim_value_usd_millions
782863,United Kingdom,Lithuania,2024-Q2,-1069.000
685031,Finland,Lithuania,2020-Q1,-673.000
775327,Finland,Finland,2023-Q4,-487.000
374900,Denmark,Lithuania,2007-Q2,-440.000
501236,Denmark,Lithuania,2012-Q3,-379.000
495220,Denmark,Lithuania,2012-Q2,-283.000
643958,Switzerland,Papua New Guinea,2018-Q3,-282.846
483188,Denmark,Lithuania,2011-Q4,-268.000
441048,Denmark,Austria,2010-Q1,-236.000
727788,France,Slovenia,2021-Q4,-191.000


## Claim-Value Cleaning Rule

The full historical dataset contains 249,073 reported numerical values. Of these, 248,839 are positive and 234 are negative. No reported observations are equal to zero.

Negative values are rare and are retained in the observed-data audit dataset because they may reflect statistical adjustments, revisions, or other BIS reporting effects. They are not treated as lending exposures.

For graph construction, only positive reported international-claim values become directed weighted edges. Missing values are not converted to zero and remain unobserved in the audit datasets.

Create audit and positive-edge datasets

In [17]:
def create_edge_datasets(dataframe):
    all_rows = dataframe.copy()

    observed_rows = dataframe[
        dataframe["claim_value_usd_millions"].notna()
    ].copy()

    positive_edges = observed_rows[
        observed_rows["claim_value_usd_millions"] > 0
    ].copy()

    negative_rows = observed_rows[
        observed_rows["claim_value_usd_millions"] < 0
    ].copy()

    return all_rows, observed_rows, positive_edges, negative_rows


full_all, full_observed, full_positive_edges, full_negative = (
    create_edge_datasets(network_full_long)
)

quarterly_all, quarterly_observed, quarterly_positive_edges, quarterly_negative = (
    create_edge_datasets(network_quarterly_long)
)

print("Full historical datasets")
print("All rows:", len(full_all))
print("Observed rows:", len(full_observed))
print("Positive edges:", len(full_positive_edges))
print("Negative rows:", len(full_negative))

print("\nRegular quarterly datasets")
print("All rows:", len(quarterly_all))
print("Observed rows:", len(quarterly_observed))
print("Positive edges:", len(quarterly_positive_edges))
print("Negative rows:", len(quarterly_negative))

Full historical datasets
All rows: 830208
Observed rows: 249073
Positive edges: 248839
Negative rows: 234

Regular quarterly datasets
All rows: 631680
Observed rows: 219231
Positive edges: 218999
Negative rows: 232


## Edge-Key Validation

A directed graph snapshot requires at most one weight for each reporting-country, counterparty-country, and period combination.

The selected BIS series definition should produce one observation for each unique directed edge at each time step. Before saving the processed data, duplicate edge keys are checked in both the full historical and regular quarterly positive-edge datasets.

If duplicates existed, they would need to be investigated and resolved before graph construction. No aggregation rule is applied unless duplicates are found.

Check duplicate edge keys

In [18]:
edge_key_columns = [
    "Reporting country",
    "Counterparty country",
    "period",
]

def duplicate_edge_report(dataframe, dataset_name):
    duplicates = dataframe[
        dataframe.duplicated(
            subset=edge_key_columns,
            keep=False
        )
    ].copy()

    print(f"{dataset_name}:")
    print("Rows:", len(dataframe))
    print("Rows involved in duplicate edge keys:", len(duplicates))

    if len(duplicates) > 0:
        display(
            duplicates[
                edge_key_columns
                + [
                    "claim_value_usd_millions",
                    "Series",
                    "period_index",
                ]
            ]
            .sort_values(edge_key_columns)
            .head(20)
        )
    else:
        print("No duplicate reporting-country / counterparty-country / period keys found.")

    print("-" * 80)


duplicate_edge_report(
    full_positive_edges,
    "Full historical positive-edge dataset"
)

duplicate_edge_report(
    quarterly_positive_edges,
    "Regular quarterly positive-edge dataset"
)

Full historical positive-edge dataset:
Rows: 248839
Rows involved in duplicate edge keys: 0
No duplicate reporting-country / counterparty-country / period keys found.
--------------------------------------------------------------------------------
Regular quarterly positive-edge dataset:
Rows: 218999
Rows involved in duplicate edge keys: 0
No duplicate reporting-country / counterparty-country / period keys found.
--------------------------------------------------------------------------------


## Stable Node Identifiers

Graph-learning libraries require numerical node identifiers rather than country names.

A single stable node index is created from the union of all reporting countries and counterparty countries in the full historical positive-edge dataset. Each country or economy receives one integer `node_id` that remains unchanged across all periods.

The same node index is applied to both datasets:

- `source_node_id` represents the reporting-country node.
- `target_node_id` represents the counterparty-country node.

This approach ensures that a country is represented consistently across graph snapshots. Countries may be absent from a particular period because no positive edge is reported, but their node ID remains stable in the full node universe.

Create node index and attach IDs

In [19]:
all_nodes = sorted(
    set(full_positive_edges["Reporting country"])
    .union(full_positive_edges["Counterparty country"])
)

node_index = pd.DataFrame({
    "node_id": range(len(all_nodes)),
    "node_name": all_nodes,
})

node_id_map = dict(
    zip(node_index["node_name"], node_index["node_id"])
)

def attach_node_ids(dataframe):
    result = dataframe.copy()

    result["source_node_id"] = (
        result["Reporting country"]
        .map(node_id_map)
        .astype("int32")
    )

    result["target_node_id"] = (
        result["Counterparty country"]
        .map(node_id_map)
        .astype("int32")
    )

    return result

full_positive_edges = attach_node_ids(full_positive_edges)
quarterly_positive_edges = attach_node_ids(quarterly_positive_edges)

print("Number of unique nodes:", len(node_index))
print("Node ID range:",
      f"{node_index['node_id'].min()} to {node_index['node_id'].max()}")

print("\nFirst 20 nodes in the stable node index:")
display(node_index.head(20))

print("\nNode-ID mapping check:")
display(
    full_positive_edges[
        [
            "Reporting country",
            "source_node_id",
            "Counterparty country",
            "target_node_id",
            "period",
            "claim_value_usd_millions",
        ]
    ].head(10)
)

print("\nUnmapped source nodes:",
      full_positive_edges["source_node_id"].isna().sum())

print("Unmapped target nodes:",
      full_positive_edges["target_node_id"].isna().sum())

Number of unique nodes: 225
Node ID range: 0 to 224

First 20 nodes in the stable node index:


,node_id,node_name
0,0,Afghanistan
1,1,Albania
2,2,Algeria
3,3,Andorra
4,4,Angola
5,5,Anguilla
6,6,Antigua and Barbuda
7,7,Argentina
8,8,Armenia
9,9,Aruba



Node-ID mapping check:


,Reporting country,source_node_id,Counterparty country,target_node_id,period,claim_value_usd_millions
187,Switzerland,191,Hong Kong SAR,88,1983-Q4,549.0
204,Switzerland,191,The Bahamas,197,1983-Q4,883.0
243,Switzerland,191,Panama,154,1983-Q4,1139.0
260,Switzerland,191,Singapore,174,1983-Q4,1411.0
272,Switzerland,191,Cayman Islands,36,1983-Q4,804.0
367,Switzerland,191,United Arab Emirates,211,1983-Q4,203.0
731,United Kingdom,212,India,91,1983-Q4,378.0
732,United Kingdom,212,Iraq,94,1983-Q4,76.0
733,United Kingdom,212,Iceland,90,1983-Q4,142.0
736,United Kingdom,212,Jamaica,99,1983-Q4,35.0



Unmapped source nodes: 0
Unmapped target nodes: 0


## Edge-Weight Transformations

The original BIS claim value is retained without modification in `claim_value_usd_millions`.

Graph neural networks can be sensitive to the extremely skewed distribution of claim values: a few very large exposures can dominate model training. Therefore, a logarithmic transformation is added:

\[
\text{log\_claim\_value} = \log(1 + \text{claim\_value\_usd\_millions})
\]

This transformation reduces the influence of extremely large values while preserving the ordering of positive exposures.

The raw claim value remains the authoritative value for interpretation, reporting, and visualization. The logarithmic value is intended only as a potential model input. No normalization is fitted at this stage, because normalization parameters must be estimated on the training period only to avoid temporal data leakage.

Add graph-ready edge fields

In [20]:
def add_edge_features(dataframe):
    result = dataframe.copy()

    result["claim_value_usd_millions"] = (
        result["claim_value_usd_millions"]
        .astype("float32")
    )

    result["log_claim_value"] = np.log1p(
        result["claim_value_usd_millions"]
    ).astype("float32")

    return result


full_positive_edges = add_edge_features(full_positive_edges)
quarterly_positive_edges = add_edge_features(quarterly_positive_edges)

print("Full historical edge-feature summary:")
display(
    full_positive_edges[
        [
            "claim_value_usd_millions",
            "log_claim_value",
        ]
    ].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

print("\nRegular quarterly edge-feature summary:")
display(
    quarterly_positive_edges[
        [
            "claim_value_usd_millions",
            "log_claim_value",
        ]
    ].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

print("\nGraph-ready edge preview:")
display(
    quarterly_positive_edges[
        [
            "period",
            "period_index",
            "Reporting country",
            "source_node_id",
            "Counterparty country",
            "target_node_id",
            "claim_value_usd_millions",
            "log_claim_value",
        ]
    ].head(10)
)

Full historical edge-feature summary:


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
claim_value_usd_millions,248839.0,5921.015137,35098.187500,0.001,0.005000,10.000000,111.369003,1295.000000,23087.904883,109713.92000,2.098198e+06
log_claim_value,248839.0,4.877602,3.039605,0.001,0.004988,2.397895,4.721788,7.167038,10.047107,11.60564,1.455659e+01



Regular quarterly edge-feature summary:


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
claim_value_usd_millions,218999.0,6574.518066,37311.464844,0.001,0.004000,9.000000,122.000000,1531.397461,26408.600000,119645.180000,2.098198e+06
log_claim_value,218999.0,4.941211,3.119333,0.001,0.003992,2.302585,4.812184,7.334589,10.181483,11.692294,1.455659e+01



Graph-ready edge preview:


,period,period_index,Reporting country,source_node_id,Counterparty country,target_node_id,claim_value_usd_millions,log_claim_value
170,2000-Q1,33,Switzerland,191,Spain,182,13482.0,9.509185
173,2000-Q1,33,Switzerland,191,France,70,31151.0,10.346634
175,2000-Q1,33,Switzerland,191,United Kingdom,212,132526.0,11.794541
187,2000-Q1,33,Switzerland,191,Hong Kong SAR,88,1744.0,7.464510
193,2000-Q1,33,Switzerland,191,Ireland,95,1616.0,7.388328
204,2000-Q1,33,Switzerland,191,The Bahamas,197,1971.0,7.586803
208,2000-Q1,33,Switzerland,191,Canada,35,4604.0,8.434898
225,2000-Q1,33,Switzerland,191,Germany,75,52303.0,10.864828
243,2000-Q1,33,Switzerland,191,Panama,154,1398.0,7.243513
260,2000-Q1,33,Switzerland,191,Singapore,174,3010.0,8.010028


The raw claim values are highly right-skewed: in the regular quarterly dataset, the median is about 122 million USD, while the maximum is about 2.10 trillion USD. The logarithmic transformation compresses this range from approximately
0.0001–2,098,198.5 to approximately 0.0001–14.56, which makes it more appropriate for later neural-network inputs.

One important detail: the transformed values are created correctly, but do not scale them yet. Scaling must be fit only on the training period later, otherwise future data could leak into model training.

## Graph-Snapshot Validation

Each BIS period will become one directed, weighted graph snapshot.

Before saving the processed datasets, snapshot-level diagnostics are calculated from positive reported edges:

- Number of directed edges.
- Number of active source nodes.
- Number of active target nodes.
- Number of unique active nodes.
- Total reported claims.
- Median reported claim.
- Maximum reported claim.

These diagnostics are not model features by themselves. They validate the graph construction and provide context when interpreting changes in network structure, model reconstruction error, or predicted systemic stress.

Build snapshot diagnostics

In [21]:
def make_snapshot_diagnostics(dataframe):
    snapshot_summary = (
        dataframe
        .groupby(
            ["period", "year", "quarter", "period_index"],
            as_index=False
        )
        .agg(
            edge_count=("claim_value_usd_millions", "size"),
            active_source_nodes=("source_node_id", "nunique"),
            active_target_nodes=("target_node_id", "nunique"),
            total_claims_usd_millions=(
                "claim_value_usd_millions",
                "sum"
            ),
            median_claim_usd_millions=(
                "claim_value_usd_millions",
                "median"
            ),
            max_claim_usd_millions=(
                "claim_value_usd_millions",
                "max"
            ),
        )
    )

    node_counts = (
        dataframe[
            [
                "period",
                "source_node_id",
                "target_node_id",
            ]
        ]
        .melt(
            id_vars="period",
            value_vars=["source_node_id", "target_node_id"],
            value_name="node_id"
        )
        .groupby("period")["node_id"]
        .nunique()
        .rename("active_node_count")
        .reset_index()
    )

    snapshot_summary = (
        snapshot_summary
        .merge(node_counts, on="period", how="left")
        .sort_values("period_index")
        .reset_index(drop=True)
    )

    return snapshot_summary


full_snapshot_diagnostics = make_snapshot_diagnostics(
    full_positive_edges
)

quarterly_snapshot_diagnostics = make_snapshot_diagnostics(
    quarterly_positive_edges
)

print("Full historical snapshot diagnostics:")
display(full_snapshot_diagnostics.head(12))

print("\nTransition snapshots:")
display(
    full_snapshot_diagnostics[
        full_snapshot_diagnostics["year"].between(1998, 2001)
    ]
)

print("\nMost recent quarterly snapshots:")
display(quarterly_snapshot_diagnostics.tail(12))

print("\nSnapshot diagnostic ranges for the regular quarterly dataset:")
display(
    quarterly_snapshot_diagnostics[
        [
            "edge_count",
            "active_node_count",
            "total_claims_usd_millions",
            "median_claim_usd_millions",
            "max_claim_usd_millions",
        ]
    ].describe().T
)

Full historical snapshot diagnostics:


,period,year,quarter,period_index,edge_count,active_source_nodes,active_target_nodes,total_claims_usd_millions,median_claim_usd_millions,max_claim_usd_millions,active_node_count
0,1983-Q4,1983,4,0,658,12,137,390591.0,70.0,24691.0,149
1,1984-Q2,1984,2,1,649,13,136,404482.0,73.0,25228.0,149
2,1984-Q4,1984,4,2,663,13,134,483966.0,86.0,25639.0,147
3,1985-Q2,1985,2,3,736,13,137,512386.0,73.5,31193.0,150
4,1985-Q4,1985,4,4,879,15,141,558437.0,58.0,35153.0,156
5,1986-Q2,1986,2,5,880,15,140,591633.0,65.5,40513.0,155
6,1986-Q4,1986,4,6,893,15,143,657826.0,70.0,59188.0,158
7,1987-Q2,1987,2,7,895,15,145,680565.0,64.0,58909.0,160
8,1987-Q4,1987,4,8,903,15,142,741782.0,69.0,75177.0,157
9,1988-Q2,1988,2,9,912,15,146,710097.0,62.0,81271.0,161



Transition snapshots:


,period,year,quarter,period_index,edge_count,active_source_nodes,active_target_nodes,total_claims_usd_millions,median_claim_usd_millions,max_claim_usd_millions,active_node_count
29,1998-Q2,1998,2,29,957,15,168,1093217.0,79.0,54724.0,183
30,1998-Q4,1998,4,30,998,15,173,1075738.0,71.5,79980.0,188
31,1999-Q2,1999,2,31,1130,15,192,4419465.0,189.5,240416.0,192
32,1999-Q4,1999,4,32,1274,16,196,4441363.0,137.5,272276.0,196
33,2000-Q1,2000,1,33,1270,16,194,4708594.0,133.5,275118.0,194
34,2000-Q2,2000,2,34,1272,16,194,4769861.0,138.5,301656.0,194
35,2000-Q3,2000,3,35,1180,16,189,4695293.0,178.5,306697.0,189
36,2000-Q4,2000,4,36,1418,18,198,4933720.0,112.5,327939.0,198
37,2001-Q1,2001,1,37,1400,18,197,5253574.0,121.5,354582.0,197
38,2001-Q2,2001,2,38,1414,18,198,5212755.0,114.0,338757.0,198



Most recent quarterly snapshots:


,period,year,quarter,period_index,edge_count,active_source_nodes,active_target_nodes,total_claims_usd_millions,median_claim_usd_millions,max_claim_usd_millions,active_node_count
93,2023-Q2,2023,2,126,2536,28,215,18310478.0,85.000000,1363076.750,215
94,2023-Q3,2023,3,127,2503,28,215,18353190.0,88.956001,1384627.875,215
95,2023-Q4,2023,4,128,2504,28,216,18976576.0,92.000000,1525848.375,216
96,2024-Q1,2024,1,129,2545,29,217,19685288.0,86.500999,1682285.125,217
97,2024-Q2,2024,2,130,2542,29,216,19732936.0,89.327499,1610884.750,216
98,2024-Q3,2024,3,131,2521,29,215,20840602.0,90.237999,1733010.750,215
99,2024-Q4,2024,4,132,2521,29,215,19649434.0,86.000000,1745280.500,215
100,2025-Q1,2025,1,133,2515,29,214,21227794.0,91.151001,1862053.000,214
101,2025-Q2,2025,2,134,2509,29,216,22447420.0,88.637001,1854820.875,216
102,2025-Q3,2025,3,135,2529,29,216,23004998.0,90.165001,1939530.750,216



Snapshot diagnostic ranges for the regular quarterly dataset:


,count,mean,std,min,25%,50%,75%,max
edge_count,105.0,2.085705e+03,4.639829e+02,1180.0,1.683000e+03,2013.0,2545.00,2638.0
active_node_count,105.0,2.091619e+02,5.960001e+00,189.0,2.060000e+02,210.0,213.00,217.0
total_claims_usd_millions,105.0,1.371250e+07,4.761038e+06,4695293.0,1.109230e+07,14712735.0,16629105.00,24441530.0
median_claim_usd_millions,105.0,1.358740e+02,5.558668e+01,75.0,9.620000e+01,109.0,167.00,287.5
max_claim_usd_millions,105.0,8.953468e+05,4.275012e+05,275118.0,5.699680e+05,854217.0,1083914.75,2098198.5


## Compact Graph-Ready Edge Tables

The positive-edge datasets retain BIS metadata for auditing. A compact graph-ready edge table is also created for efficient loading in later graph-learning notebooks.

Each row represents one positive directed banking exposure in one period. The compact table retains:

- Original period label and time fields.
- Stable integer source and target node identifiers.
- Reporting-country and counterparty-country names for interpretation.
- Original claim amount in millions of US dollars.
- Log-transformed claim value for potential model input.

The original BIS metadata remains available in the larger audit datasets. The compact edge tables are derived files intended for graph construction and model input.

Create compact edge tables

In [22]:
graph_edge_columns = [
    "period",
    "year",
    "quarter",
    "period_index",
    "is_quarterly_era",
    "is_semiannual_era",
    "Reporting country",
    "source_node_id",
    "Counterparty country",
    "target_node_id",
    "claim_value_usd_millions",
    "log_claim_value",
]

full_graph_edges = (
    full_positive_edges[graph_edge_columns]
    .sort_values(
        ["period_index", "source_node_id", "target_node_id"]
    )
    .reset_index(drop=True)
)

quarterly_graph_edges = (
    quarterly_positive_edges[graph_edge_columns]
    .sort_values(
        ["period_index", "source_node_id", "target_node_id"]
    )
    .reset_index(drop=True)
)

print("Full historical graph-edge table shape:", full_graph_edges.shape)
print("Regular quarterly graph-edge table shape:", quarterly_graph_edges.shape)

print("\nFull historical graph-edge preview:")
display(full_graph_edges.head(10))

print("\nRegular quarterly graph-edge preview:")
display(quarterly_graph_edges.head(10))

print("\nMissing values in compact quarterly edge table:")
display(
    quarterly_graph_edges.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

Full historical graph-edge table shape: (248839, 12)
Regular quarterly graph-edge table shape: (218999, 12)

Full historical graph-edge preview:


,period,year,quarter,period_index,is_quarterly_era,is_semiannual_era,Reporting country,source_node_id,Counterparty country,target_node_id,claim_value_usd_millions,log_claim_value
0,1983-Q4,1983,4,0,False,True,Austria,11,Algeria,2,256.0,5.549076
1,1983-Q4,1983,4,0,False,True,Austria,11,Angola,4,1.0,0.693147
2,1983-Q4,1983,4,0,False,True,Austria,11,Argentina,7,189.0,5.247024
3,1983-Q4,1983,4,0,False,True,Austria,11,Australia,10,15.0,2.772589
4,1983-Q4,1983,4,0,False,True,Austria,11,Bahrain,13,263.0,5.575949
5,1983-Q4,1983,4,0,False,True,Austria,11,Bangladesh,14,5.0,1.791759
6,1983-Q4,1983,4,0,False,True,Austria,11,Barbados,15,26.0,3.295837
7,1983-Q4,1983,4,0,False,True,Austria,11,Bermuda,20,18.0,2.944439
8,1983-Q4,1983,4,0,False,True,Austria,11,Bolivia,22,46.0,3.850147
9,1983-Q4,1983,4,0,False,True,Austria,11,Brazil,26,213.0,5.365976



Regular quarterly graph-edge preview:


,period,year,quarter,period_index,is_quarterly_era,is_semiannual_era,Reporting country,source_node_id,Counterparty country,target_node_id,claim_value_usd_millions,log_claim_value
0,2000-Q1,2000,1,33,True,False,Austria,11,Albania,1,2.0,1.098612
1,2000-Q1,2000,1,33,True,False,Austria,11,Algeria,2,759.0,6.633318
2,2000-Q1,2000,1,33,True,False,Austria,11,Andorra,3,9.0,2.302585
3,2000-Q1,2000,1,33,True,False,Austria,11,Angola,4,30.0,3.433987
4,2000-Q1,2000,1,33,True,False,Austria,11,Argentina,7,449.0,6.109248
5,2000-Q1,2000,1,33,True,False,Austria,11,Australia,10,824.0,6.715384
6,2000-Q1,2000,1,33,True,False,Austria,11,Azerbaijan,12,3.0,1.386294
7,2000-Q1,2000,1,33,True,False,Austria,11,Bahrain,13,184.0,5.220356
8,2000-Q1,2000,1,33,True,False,Austria,11,Barbados,15,1.0,0.693147
9,2000-Q1,2000,1,33,True,False,Austria,11,Belarus,16,34.0,3.555348



Missing values in compact quarterly edge table:


,missing_count
period,0
year,0
quarter,0
period_index,0
is_quarterly_era,0
is_semiannual_era,0
Reporting country,0
source_node_id,0
Counterparty country,0
target_node_id,0


One important correction

For graph autoencoders and temporal graph neural networks, we should use the regular quarterly graph-edge table as the primary modeling input:

`quarterly_graph_edges`

It has 105 regularly spaced snapshots from 2000-Q1 to 2026-Q1. The full historical table remains valuable for descriptive analysis and potential separate semiannual modeling, but it should not be mixed into a quarterly TGN sequence without explicitly handling irregular time intervals.

## Data Dictionary

A data dictionary is created for the compact graph-edge tables. It records each variable’s meaning, data type, source, and role in graph modeling.

The dictionary distinguishes original BIS variables from derived preprocessing fields. It also clarifies that the raw and log-transformed claim values have different purposes:

- `claim_value_usd_millions` is the original reported BIS value.
- `log_claim_value` is a derived feature intended for model input.

Create the data dictionary

In [23]:
data_dictionary = pd.DataFrame([
    {
        "column_name": "period",
        "data_type": "string",
        "source": "BIS / derived",
        "description": "Original BIS observation-period label, such as 2000-Q1.",
        "model_role": "Temporal ordering and graph snapshot identifier",
    },
    {
        "column_name": "year",
        "data_type": "integer",
        "source": "Derived from period",
        "description": "Calendar year extracted from the BIS period label.",
        "model_role": "Temporal grouping and chronological splitting",
    },
    {
        "column_name": "quarter",
        "data_type": "integer",
        "source": "Derived from period",
        "description": "Quarter number extracted from the BIS period label.",
        "model_role": "Temporal grouping",
    },
    {
        "column_name": "period_index",
        "data_type": "integer",
        "source": "Derived from BIS period order",
        "description": "Sequential index following the order of periods in the BIS source.",
        "model_role": "Chronological graph-snapshot ordering",
    },
    {
        "column_name": "is_quarterly_era",
        "data_type": "boolean",
        "source": "Derived from period",
        "description": "True for observations from 2000-Q1 onward.",
        "model_role": "Frequency-regime indicator",
    },
    {
        "column_name": "is_semiannual_era",
        "data_type": "boolean",
        "source": "Derived from period",
        "description": "True for observations before 2000-Q1, when source coverage is primarily semiannual.",
        "model_role": "Frequency-regime indicator",
    },
    {
        "column_name": "Reporting country",
        "data_type": "string",
        "source": "BIS",
        "description": "Country of the reporting banking system; the source node in the directed network.",
        "model_role": "Interpretability and source-node label",
    },
    {
        "column_name": "source_node_id",
        "data_type": "integer",
        "source": "Derived from stable node index",
        "description": "Stable integer identifier for the reporting-country node.",
        "model_role": "Graph source-node index",
    },
    {
        "column_name": "Counterparty country",
        "data_type": "string",
        "source": "BIS",
        "description": "Country or economy of the direct counterparty; the target node in the directed network.",
        "model_role": "Interpretability and target-node label",
    },
    {
        "column_name": "target_node_id",
        "data_type": "integer",
        "source": "Derived from stable node index",
        "description": "Stable integer identifier for the counterparty-country node.",
        "model_role": "Graph target-node index",
    },
    {
        "column_name": "claim_value_usd_millions",
        "data_type": "float",
        "source": "BIS",
        "description": "Positive reported international claim amount, in millions of US dollars.",
        "model_role": "Original directed edge weight",
    },
    {
        "column_name": "log_claim_value",
        "data_type": "float",
        "source": "Derived",
        "description": "Natural logarithm of one plus the positive claim value.",
        "model_role": "Potential edge feature for neural-network input; scale later using training data only",
    },
])

display(data_dictionary)

,column_name,data_type,source,description,model_role
0,period,string,BIS / derived,"Original BIS observation-period label, such as 2000-Q1.",Temporal ordering and graph snapshot identifier
1,year,integer,Derived from period,Calendar year extracted from the BIS period label.,Temporal grouping and chronological splitting
2,quarter,integer,Derived from period,Quarter number extracted from the BIS period label.,Temporal grouping
3,period_index,integer,Derived from BIS period order,Sequential index following the order of periods in the BIS source.,Chronological graph-snapshot ordering
4,is_quarterly_era,boolean,Derived from period,True for observations from 2000-Q1 onward.,Frequency-regime indicator
5,is_semiannual_era,boolean,Derived from period,"True for observations before 2000-Q1, when source coverage is primarily semiannual.",Frequency-regime indicator
6,Reporting country,string,BIS,Country of the reporting banking system; the source node in the directed network.,Interpretability and source-node label
7,source_node_id,integer,Derived from stable node index,Stable integer identifier for the reporting-country node.,Graph source-node index
8,Counterparty country,string,BIS,Country or economy of the direct counterparty; the target node in the directed network.,Interpretability and target-node label
9,target_node_id,integer,Derived from stable node index,Stable integer identifier for the counterparty-country node.,Graph target-node index


## Saving Processed Outputs

The final processed files are saved in `data/processed/`.

The regular quarterly graph-edge file is the primary input for Project Three temporal graph models. It covers `2000-Q1` through `2026-Q1` and contains 105 regularly spaced graph snapshots.

The full historical graph-edge file is retained for long-run descriptive analysis, robustness checks, and separate models that explicitly handle the semiannual historical period.

The audit files preserve missing and negative observations so that all graph-construction decisions remain traceable to the original BIS source.

In [24]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files_to_save = {
    "bis_cbs_full_1983Q4_2026Q1_all_rows.csv": full_all,
    "bis_cbs_full_1983Q4_2026Q1_observed.csv": full_observed,
    "bis_cbs_full_1983Q4_2026Q1_positive_edges.csv": full_positive_edges,
    "bis_cbs_full_1983Q4_2026Q1_graph_edges.csv": full_graph_edges,
    "bis_cbs_quarterly_2000Q1_2026Q1_all_rows.csv": quarterly_all,
    "bis_cbs_quarterly_2000Q1_2026Q1_observed.csv": quarterly_observed,
    "bis_cbs_quarterly_2000Q1_2026Q1_positive_edges.csv": quarterly_positive_edges,
    "bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv": quarterly_graph_edges,
    "bis_cbs_node_index.csv": node_index,
    "bis_cbs_full_period_quality.csv": full_period_quality,
    "bis_cbs_quarterly_period_quality.csv": quarterly_period_quality,
    "bis_cbs_full_snapshot_diagnostics.csv": full_snapshot_diagnostics,
    "bis_cbs_quarterly_snapshot_diagnostics.csv": quarterly_snapshot_diagnostics,
    "bis_cbs_graph_edge_data_dictionary.csv": data_dictionary,
}

for filename, dataframe in files_to_save.items():
    dataframe.to_csv(PROCESSED_DIR / filename, index=False)

print("Saved processed files:")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"- {path.name}: {size_mb:,.2f} MB")

Saved processed files:
- bis_cbs_full_1983Q4_2026Q1_all_rows.csv: 189.15 MB
- bis_cbs_full_1983Q4_2026Q1_graph_edges.csv: 16.96 MB
- bis_cbs_full_1983Q4_2026Q1_observed.csv: 57.45 MB
- bis_cbs_full_1983Q4_2026Q1_positive_edges.csv: 61.34 MB
- bis_cbs_full_period_quality.csv: 0.01 MB
- bis_cbs_full_snapshot_diagnostics.csv: 0.01 MB
- bis_cbs_graph_edge_data_dictionary.csv: 0.00 MB
- bis_cbs_node_index.csv: 0.00 MB
- bis_cbs_quarterly_2000Q1_2026Q1_all_rows.csv: 144.18 MB
- bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv: 14.98 MB
- bis_cbs_quarterly_2000Q1_2026Q1_observed.csv: 50.62 MB
- bis_cbs_quarterly_2000Q1_2026Q1_positive_edges.csv: 54.04 MB
- bis_cbs_quarterly_period_quality.csv: 0.01 MB
- bis_cbs_quarterly_snapshot_diagnostics.csv: 0.01 MB


# Preprocessing Conclusion

This notebook transformed the BIS consolidated banking statistics data into reproducible, graph-ready directed edge tables for the international banking network.

## Primary Modeling Dataset

The primary dataset for Project Three is:

```text
data/processed/bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv
```

This file contains one positive directed cross-border banking exposure per row. It covers the regular quarterly reporting period from `2000-Q1` through `2026-Q1`.

Key properties:

- 218,999 positive directed edges.
- 105 regularly spaced quarterly graph snapshots.
- Stable integer node identifiers shared across all periods.
- Directed edges from reporting banking-system countries to counterparty countries.
- Original claim values in millions of US dollars.
- Log-transformed claim values for potential neural-network input.
- No missing values in the compact graph-ready table.

## Files Produced

The processed directory contains four categories of outputs:

1. Graph-ready edge tables:
   - `bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv`
   - `bis_cbs_full_1983Q4_2026Q1_graph_edges.csv`

2. Audit tables:
   - Full and quarterly all-row datasets.
   - Full and quarterly observed-value datasets.
   - Full and quarterly positive-edge datasets.

3. Diagnostics:
   - Period-quality reports showing observed, missing, zero, and negative values.
   - Snapshot diagnostics showing node counts, edge counts, and claim distributions by period.

4. Metadata:
   - `bis_cbs_node_index.csv`
   - `bis_cbs_graph_edge_data_dictionary.csv`

## Modeling Decisions

The graph is represented as a directed weighted network:

\[
\text{Reporting country} \rightarrow \text{Counterparty country}
\]

An edge exists when the BIS reports a strictly positive international claim value. The edge weight is the claim value in millions of US dollars.

Missing values are retained in audit tables but are not interpreted as zero-valued claims. Zero and negative values are excluded from the positive-edge graph because the graph-modeling dataset represents observed positive exposures.

The natural-log transformation is defined as:

\[
\text{log_claim_value} = \log(1 + \text{claim_value_usd_millions})
\]

This reduces the influence of extremely large claims during model training. Any additional scaling or normalization must be fitted only on the training period to prevent temporal data leakage.

## Frequency Regimes and Temporal Splitting

The BIS history contains two reporting-frequency regimes:

- Before `2000-Q1`, observations are primarily semiannual.
- From `2000-Q1` onward, observations are regularly quarterly.

The primary temporal graph-learning experiments should use only the regular quarterly period. The longer historical dataset remains available for descriptive analysis, robustness checks, or models that explicitly account for irregular time intervals.

Training, validation, and test data must be split chronologically rather than randomly. A recommended initial split is:

- Training: `2000-Q1` through `2018-Q4`
- Validation: `2019-Q1` through `2021-Q4`
- Test: `2022-Q1` through `2026-Q1`

## Recommended Next Steps

The next notebook should load the quarterly graph-edge table and construct one directed weighted graph snapshot per quarter.

Subsequent steps should include:

1. Verify that each snapshot contains only valid node identifiers and positive edge weights.
2. Build sparse adjacency representations or PyTorch Geometric edge-index tensors.
3. Create chronological train, validation, and test splits.
4. Fit edge-feature scaling using only training snapshots.
5. Train a baseline graph autoencoder for link reconstruction or anomaly scoring.
6. Extend the baseline to a temporal graph neural network for dynamic link prediction and anomaly detection.
7. Evaluate results against snapshot diagnostics and major changes in network coverage.

## Important adjustment

Use the following exact filenames when you later load data:

| Purpose | File |
|---|---|
| Primary temporal-model input | `bis_cbs_quarterly_2000Q1_2026Q1_graph_edges.csv` |
| Long-run historical analysis | `bis_cbs_full_1983Q4_2026Q1_graph_edges.csv` |
| Stable mapping from integer IDs to countries | `bis_cbs_node_index.csv` |
| Field definitions | `bis_cbs_graph_edge_data_dictionary.csv` |
| Per-quarter coverage and data-quality checks | `bis_cbs_quarterly_period_quality.csv` |
| Per-quarter network diagnostics | `bis_cbs_quarterly_snapshot_diagnostics.csv` |

Note:

Before modeling, retain the raw claim_value_usd_millions for interpretability and reporting, but use a training-set-fitted standardized version of log_claim_value as the numerical edge feature. That approach limits the effect of the very large upper-tail exposures without leaking information from future quarters.

## Important Limitations to Acknowledge

The data are well suited to the analysis, but it **does not** represent.

- **Scope of the data:** The graph represents international claims reported through BIS reporting banking systems. It does not capture every financial transaction or exposure worldwide.

- **Changes in reporting coverage:** BIS reporting-country coverage has expanded over time. As a result, increases in the number of nodes or edges may partly reflect broader reporting coverage rather than genuine growth in the underlying economic network.

- **Missing values vs. zero exposure:** A missing value should **not** automatically be interpreted as a zero exposure. The audit files preserve this distinction and should be used when investigating missing observations.

- **Positive-edge representation:** The positive-edge table intentionally excludes zero and negative values. This is appropriate for constructing a positive-exposure graph, but the exclusion should be explicitly documented.

- **Country and territorial definitions:** Country names and territorial definitions may have changed across decades. The stable node index should therefore be used consistently. Unusual or ambiguous nodes should be reviewed before drawing country-level conclusions.

- **Latest data availability:** The latest available observations end in **2026-Q1**. Findings should not be interpreted as extending beyond this data snapshot.

- **Interpretation of anomaly scores:** An anomaly score indicates that an observation is **unusual relative to the model and the observed historical structure**. It is not, by itself, evidence of misconduct, financial distress, or a data-quality error.

## Recommended Quality Checks Before Training

The data-cleaning stage is complete. Before starting model training in the next notebook, run the following **model-readiness checks**:

1. **Validate edge weights**  
   Confirm that every quarter contains only positive edge weights in the positive-exposure graph.

2. **Validate node IDs**  
   Confirm that every `source_node_id` and `target_node_id` falls within the valid range of the stable node index.

3. **Inspect self-loops**  
   Count self-loops and determine whether they should be retained or removed based on the BIS interpretation of same-country reporting/counterparty relationships.

4. **Check for duplicate edges**  
   Verify that there are no duplicate `(period, source_node_id, target_node_id)` combinations after aggregation and preprocessing.

5. **Calculate network diagnostics**  
   For each quarter, calculate:
   - Network density
   - Total claims
   - Median claim size
   - Number of active nodes

   Use these measures as contextual diagnostics when interpreting model results.

6. **Prevent data leakage**  
   Fit all scalers, encoders, and thresholds using **training data only**. Do not use information from the validation or test periods during fitting.

7. **Protect the test set**  
   Keep the test set completely untouched until the final model evaluation. It should be used only once the model-development process is complete.